# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Display dataset title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets present in the dataset with their @id and names
record_sets = list(dataset.record_sets)

print("Available record sets:")
for record_set in record_sets:
    print(f"@id: {record_set['@id']}")
    print(f"Name: {record_set.get('name', 'N/A')}")
    print(f"Description: {record_set.get('description', 'N/A')}")
    print('---')
    # List fields for each record set
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        field_id = field.get('@id', '')
        field_name = field.get('name', 'N/A')
        print(f"    - @id: {field_id}, name: {field_name}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Example extraction for all available record sets
# Edit this cell with the `@id`s from the data overview printed above for your specific use case

# List all record set @id
record_set_ids = [record_set['@id'] for record_set in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)

# Display columns from the first data frame loaded, if any
if len(dataframes) > 0:
    first_record_set_id = list(dataframes.keys())[0]
    print(f"Columns for record set {first_record_set_id}:\n", dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())
else:
    print("No dataframes loaded from record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Sample: Explore and filter a numeric field
# Replace values of record_set_id, numeric_field, group_field as needed based on the columns output in Section 3

if len(dataframes) > 0:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id].copy()

    # Infer a numeric field (e.g. 'Molecular Weight' or similar column, adapt as necessary)
    # For this dataset, possible numeric fields could be 'MW', 'PeptideCount', 'Coverage(%)', etc.
    candidate_numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] or pd.api.types.is_numeric_dtype(df[col])]
    if len(candidate_numeric_fields) == 0:
        print("No numeric fields found for EDA.")
    else:
        numeric_field = candidate_numeric_fields[0]

        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
        if threshold == 0: threshold = 10

        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Pick the first object/categorical field for grouping
        group_field_candidates = [col for col in df.columns if df[col].dtype == 'object']
        if len(group_field_candidates) > 0:
            group_field = group_field_candidates[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
else:
    print("No data available for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Pick a numeric field for visualization
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_cols:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_cols[0]], kde=True)
        plt.title(f"Distribution of {numeric_cols[0]} values")
        plt.xlabel(numeric_cols[0])
        plt.ylabel("Frequency")
        plt.show()
    else:
        print("No numeric columns found to plot.")
else:
    print("No data loaded for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Croissant dataset was loaded and its structure explored using `mlcroissant`.
- Key record sets and their fields were listed by `@id`, and tabular data was extracted for further processing.
- Exploratory Data Analysis included filtering and normalization of numeric fields, and grouping by categorical metadata.
- Visualizations provided insight into the distribution of selected numeric features.

Users may continue to explore and analyze the dataset by specifying different record sets and fields by their `@id` per the overview above.